# preprocessing.py

Shared preprocessing pipeline for all Heart Disease prediction models.
Run this once; the preprocessed artefacts are saved to disk and reloaded
by each model script.

In [2]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.utils.class_weight import compute_class_weight
import joblib
import os

## 1. Load dataset

In [4]:
DATA_PATH = "dataset_under_40.csv"   # update path if needed
OUT_DIR   = "preprocessed"
os.makedirs(OUT_DIR, exist_ok=True)

data = pd.read_csv(DATA_PATH)
print(f"Raw shape : {data.shape}")

Raw shape : (73209, 19)


## 2. Remove duplicates

In [6]:
dup = data.duplicated().sum()
print(f"Duplicates removed : {dup}")
data = data.drop_duplicates()

Duplicates removed : 24


## 3. Derived / engineered features (user-defined)

In [8]:
# BMI Category
data['BMI_Category'] = pd.cut(
    data['BMI'],
    bins=[0, 18.5, 24.9, 29.9, np.inf],
    labels=['Underweight', 'Normal weight', 'Overweight', 'Obesity']
)

# Health Checkup Frequency
checkup_mapping = {
    'Within the past year'    : 4,
    'Within the past 2 years' : 2,
    'Within the past 5 years' : 1,
    '5 or more years ago'     : 0.2,
    'Never'                   : 0
}
data['Checkup_Frequency'] = data['Checkup'].replace(checkup_mapping)

exercise_mapping = {'Yes': 1, 'No': 0}
smoking_mapping  = {'Yes': -1, 'No': 0}

# Lifestyle Score
data['Lifestyle_Score'] = (
    data['Exercise'].replace(exercise_mapping)
    - data['Smoking_History'].replace(smoking_mapping)
    + data['Fruit_Consumption'] / 10
    + data['Green_Vegetables_Consumption'] / 10
    - data['Alcohol_Consumption'] / 10
)

# Healthy Diet Score
data['Healthy_Diet_Score'] = (
    data['Fruit_Consumption'] / 10
    + data['Green_Vegetables_Consumption'] / 10
    - data['FriedPotato_Consumption'] / 10
)

# Interaction terms
data['Smoking_Alcohol']       = data['Smoking_History'].replace(smoking_mapping) * data['Alcohol_Consumption']
data['Checkup_Exercise']      = data['Checkup_Frequency'] * data['Exercise'].replace(exercise_mapping)
data['Height_to_Weight']      = data['Height_(cm)'] / data['Weight_(kg)']
data['Fruit_Vegetables']      = data['Fruit_Consumption'] * data['Green_Vegetables_Consumption']
data['HealthyDiet_Lifestyle'] = data['Healthy_Diet_Score'] * data['Lifestyle_Score']
data['Alcohol_FriedPotato']   = data['Alcohol_Consumption'] * data['FriedPotato_Consumption']

# Additional engineered features (supplement to user list)
# Risk composite: high BMI + low exercise + smoking
data['Risk_Composite'] = data['BMI'] / 25 + data['Smoking_History'].replace({'Yes': 1, 'No': 0}) - data['Exercise'].replace(exercise_mapping)
# Alcohol per unit weight
data['Alcohol_per_Weight'] = data['Alcohol_Consumption'] / (data['Weight_(kg)'] + 1)

C:\Users\Sanjay Vikas\AppData\Local\Temp\ipykernel_12240\1539117083.py:16: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  data['Checkup_Frequency'] = data['Checkup'].replace(checkup_mapping)
C:\Users\Sanjay Vikas\AppData\Local\Temp\ipykernel_12240\1539117083.py:23: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  data['Exercise'].replace(exercise_mapping)
C:\Users\Sanjay Vikas\AppData\Local\Temp\ipykernel_12240\1539117083.py:24: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To 

## 4. Encode target: Diabetes mapping first

In [10]:
diabetes_mapping = {
    'No'                                          : 0,
    'No, pre-diabetes or borderline diabetes'     : 0,
    'Yes, but female told only during pregnancy'  : 1,
    'Yes'                                         : 1
}
data['Diabetes'] = data['Diabetes'].map(diabetes_mapping)

## 5. Binary columns

In [12]:
binary_columns = [
    'Heart_Disease', 'Skin_Cancer', 'Other_Cancer',
    'Depression', 'Arthritis', 'Smoking_History', 'Exercise'
]
for col in binary_columns:
    data[col] = data[col].map({'Yes': 1, 'No': 0})

## 6. Ordinal encodings

In [14]:
general_health_mapping = {'Poor': 0, 'Fair': 1, 'Good': 2, 'Very Good': 3, 'Excellent': 4}
data['General_Health'] = data['General_Health'].map(general_health_mapping)

bmi_cat_mapping = {'Underweight': 0, 'Normal weight': 1, 'Overweight': 2, 'Obesity': 3}
data['BMI_Category'] = data['BMI_Category'].map(bmi_cat_mapping).astype(int)

age_category_mapping = {
    '18-24': 0, '25-29': 1, '30-34': 2, '35-39': 3,
    '40-44': 4, '45-49': 5, '50-54': 6, '55-59': 7,
    '60-64': 8, '65-69': 9, '70-74': 10, '75-79': 11, '80+': 12
}
data['Age_Category'] = data['Age_Category'].map(age_category_mapping)

## 7. One-hot encode Sex; drop Checkup raw

In [16]:
data = pd.get_dummies(data, columns=['Sex'])
data = data.drop(columns=['Checkup'])

## 8. Handle remaining NaN (from mapping misses)

In [18]:
print(f"Missing values per column:\n{data.isnull().sum()[data.isnull().sum() > 0]}")
data = data.fillna(data.median(numeric_only=True))

Missing values per column:
Series([], dtype: int64)


## 9. Target: multi-class cardiovascular risk Low=0, Moderate=1, High=2 Built from Heart_Disease + clinical signals

In [20]:
def assign_risk(row):
    score = 0
    score += row['Heart_Disease']          * 3
    score += row['Diabetes']               * 1
    score += row['Smoking_History']        * 1
    score += (1 - row['Exercise'])         * 0.5
    score += row['Depression']             * 0.5
    score += (row['BMI_Category'] >= 2)    * 1
    score += row['Alcohol_Consumption'] > 14
    if score >= 4:
        return 2   # High
    elif score >= 2:
        return 1   # Moderate
    else:
        return 0   # Low

data['Risk_Level'] = data.apply(assign_risk, axis=1)
print("Risk_Level distribution:\n", data['Risk_Level'].value_counts())

# Also keep binary Heart_Disease for reference
target_binary = data['Heart_Disease']
target_multi  = data['Risk_Level']

Risk_Level distribution:
 Risk_Level
0    51123
1    21154
2      908
Name: count, dtype: int64


## 10. Feature matrix

In [22]:
drop_cols = ['Heart_Disease', 'Risk_Level']
X = data.drop(columns=drop_cols)
y = target_multi          # change to target_binary for binary task

feature_names = list(X.columns)
print(f"Final feature count : {len(feature_names)}")

Final feature count : 30


## 11. Train / Validation / Test split  70/15/15

In [24]:
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, random_state=42, stratify=y
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=42, stratify=y_temp
)
print(f"Train {X_train.shape} | Val {X_val.shape} | Test {X_test.shape}")

Train (51229, 30) | Val (10978, 30) | Test (10978, 30)


## 12. Scale numerical features

In [26]:
numerical_cols = [
    'Height_(cm)', 'Weight_(kg)', 'BMI', 'Alcohol_Consumption',
    'Fruit_Consumption', 'Green_Vegetables_Consumption', 'FriedPotato_Consumption',
    'Checkup_Frequency', 'Lifestyle_Score', 'Healthy_Diet_Score',
    'Smoking_Alcohol', 'Checkup_Exercise', 'Height_to_Weight',
    'Fruit_Vegetables', 'HealthyDiet_Lifestyle', 'Alcohol_FriedPotato',
    'Risk_Composite', 'Alcohol_per_Weight'
]
num_cols_present = [c for c in numerical_cols if c in X.columns]

scaler = StandardScaler()
X_train[num_cols_present] = scaler.fit_transform(X_train[num_cols_present])
X_val  [num_cols_present] = scaler.transform(X_val[num_cols_present])
X_test [num_cols_present] = scaler.transform(X_test[num_cols_present])

## 13. Class weights (for imbalanced target)

In [28]:
classes = np.unique(y_train)
cw = compute_class_weight('balanced', classes=classes, y=y_train)
class_weights = dict(zip(classes, cw))
print("Class weights:", class_weights)

Class weights: {0: 0.47717915758490287, 1: 1.1531829641635152, 2: 26.89186351706037}


## 14. Save artefacts

In [30]:
joblib.dump(scaler,        f"{OUT_DIR}/scaler.pkl")
joblib.dump(feature_names, f"{OUT_DIR}/feature_names.pkl")
joblib.dump(class_weights, f"{OUT_DIR}/class_weights.pkl")

X_train.to_csv(f"{OUT_DIR}/X_train.csv", index=False)
X_val  .to_csv(f"{OUT_DIR}/X_val.csv",   index=False)
X_test .to_csv(f"{OUT_DIR}/X_test.csv",  index=False)
y_train.to_csv(f"{OUT_DIR}/y_train.csv", index=False)
y_val  .to_csv(f"{OUT_DIR}/y_val.csv",   index=False)
y_test .to_csv(f"{OUT_DIR}/y_test.csv",  index=False)

print("\n✅ Preprocessing complete – artefacts saved to ./preprocessed/")


✅ Preprocessing complete – artefacts saved to ./preprocessed/
